# Experiment 1.3a-i: Effective Rank Evolution — Layer-by-Layer vs Aggregate

## Counterpart identity

This notebook is the explanatory counterpart to:

- `run_experiment.py`
- path: `experiments/Tier_1_Core_Mechanism_Tests/Exp_1.3_Singular_Value_Spectrum_Evolution/1.3a_Spectrum_Effective_Rank_Over_Training/1.3a-i_Effective_Rank_Layer-by-Layer_vs_Aggregate/run_experiment.py`

The notebook intentionally **uses the script's returned results instead of re-implementing the experiment core**. The extra notebook content is for reproducibility, visualization, and interpretation.

## Honest scope of this first-pass study

This is a **deterministic single-seed toy trajectory study** of a 6-layer, 32×32 deep linear network. It measures:

- per-layer effective rank over training,
- product-matrix effective rank over training,
- per-layer condition number over training,
- product-matrix condition number over training,
- loss trajectories.

It does **not** directly test depth scaling laws, seed-to-seed uncertainty, or statistical generalization. Any interpretation below should therefore be read as a **single-run mechanistic case study** rather than a population-level claim.


In [ ]:
from __future__ import annotations

import importlib.util
import platform
import sys
import time
from html import escape
from pathlib import Path

import matplotlib
import numpy as np

IN_NOTEBOOK = False
try:
    IN_NOTEBOOK = get_ipython().__class__.__name__ == "ZMQInteractiveShell"  # type: ignore[name-defined]
except Exception:
    IN_NOTEBOOK = False

if not IN_NOTEBOOK:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt

try:
    from IPython.display import HTML, Markdown, display
except Exception:
    HTML = None
    Markdown = None

    def display(obj):
        print(obj)


EXPERIMENT_RELATIVE = Path(
    "experiments/Tier_1_Core_Mechanism_Tests/Exp_1.3_Singular_Value_Spectrum_Evolution/"
    "1.3a_Spectrum_Effective_Rank_Over_Training/"
    "1.3a-i_Effective_Rank_Layer-by-Layer_vs_Aggregate"
)


def resolve_experiment_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = []
    for base in [cwd, *cwd.parents]:
        candidates.append(base)
        candidates.append(base / EXPERIMENT_RELATIVE)

    for candidate in candidates:
        if (candidate / "run_experiment.py").exists() and (candidate / "run_experiment.ipynb").exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not locate the experiment directory containing run_experiment.py and run_experiment.ipynb."
    )


def import_module_from_path(path: Path, module_name: str):
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    assert spec is not None and spec.loader is not None
    spec.loader.exec_module(module)
    return module


def format_value(value, digits: int = 3) -> str:
    if isinstance(value, (np.floating, float)):
        if np.isnan(value):
            return "nan"
        if np.isposinf(value):
            return "inf"
        if np.isneginf(value):
            return "-inf"
        magnitude = abs(float(value))
        if magnitude != 0 and (magnitude >= 1e4 or magnitude < 1e-2):
            return f"{float(value):.{digits}e}"
        return f"{float(value):.{digits}f}"
    if isinstance(value, (np.integer, int)):
        return str(int(value))
    return str(value)


def show_markdown(text: str) -> None:
    if Markdown is not None:
        display(Markdown(text))
    else:
        print(text)


def show_table(rows, columns=None, title: str | None = None) -> None:
    if not rows:
        print(f"{title or 'Table'}: <empty>")
        return

    if columns is None:
        columns = list(rows[0].keys())

    if title:
        show_markdown(f"### {title}")

    formatted_rows = []
    for row in rows:
        formatted_rows.append({col: format_value(row.get(col, "")) for col in columns})

    if HTML is not None:
        header_html = "".join(
            f"<th style='text-align:left;padding:6px 8px;border-bottom:1px solid #bbb'>{escape(str(col))}</th>"
            for col in columns
        )
        body_parts = []
        for row in formatted_rows:
            body_parts.append(
                "<tr>"
                + "".join(
                    f"<td style='text-align:left;padding:6px 8px;border-bottom:1px solid #eee'>{escape(str(row[col]))}</td>"
                    for col in columns
                )
                + "</tr>"
            )
        html = (
            "<table style='border-collapse:collapse;font-family:monospace;font-size:13px'>"
            f"<thead><tr>{header_html}</tr></thead>"
            f"<tbody>{''.join(body_parts)}</tbody></table>"
        )
        display(HTML(html))
    else:
        widths = {col: max(len(str(col)), max(len(row[col]) for row in formatted_rows)) for col in columns}
        header = " | ".join(f"{col:<{widths[col]}}" for col in columns)
        sep = "-+-".join("-" * widths[col] for col in columns)
        print(header)
        print(sep)
        for row in formatted_rows:
            print(" | ".join(f"{row[col]:<{widths[col]}}" for col in columns))


def test_rows_to_table(tests_dict):
    rows = []
    for label in ["T1", "T2", "T3", "T4"]:
        entry = tests_dict[label]
        rows.append(
            {
                "test": label,
                "passed": "PASS" if entry["passed"] else "FAIL",
                "description": entry["description"],
                "observed": entry["observed_summary"],
                "interpretation": entry["interpretation"],
            }
        )
    return rows


EXPERIMENT_DIR = resolve_experiment_dir()
SCRIPT_PATH = EXPERIMENT_DIR / "run_experiment.py"
NOTEBOOK_PATH = EXPERIMENT_DIR / "run_experiment.ipynb"


## Load the counterpart script and execute the exact experiment core

The next cell imports `run_experiment.py` as a module and calls its `run_experiment(...)` function directly. This keeps the script and notebook aligned in this first completion pass.


In [ ]:
experiment_module = import_module_from_path(SCRIPT_PATH, "exp_1_3a_i_run_experiment")

notebook_start = time.perf_counter()
results = experiment_module.run_experiment(
    save_plots=True,
    save_results=True,
    output_dir=EXPERIMENT_DIR,
    verbose=False,
)
notebook_elapsed = time.perf_counter() - notebook_start

config = results["config"]
problem = results["problem_summary"]
lr_search = results["lr_search"]
optimizers = results["optimizers"]
results_sgd = optimizers["sgd"]
results_muon = optimizers["muon"]
report_rows = results["report_rows"]
final_summary = results["final_summary"]
tests = results["tests"]
artifacts = results["artifacts"]

print(f"Imported script: {SCRIPT_PATH}")
print(f"Notebook path:  {NOTEBOOK_PATH}")
print(f"Script-reported runtime: {results['runtime_seconds']:.2f} s")
print(f"Notebook wall time for run_experiment(): {notebook_elapsed:.2f} s")
print(f"Overall verdict returned by script: {tests['overall']} ({tests['tests_passed']}/{tests['tests_total']})")


## Reproducibility, runtime, and configuration

This section logs the exact script counterpart, runtime, environment, and experimental configuration used for the run shown in the rest of the notebook.


In [ ]:
repro_rows = [
    {"field": "script counterpart", "value": str(SCRIPT_PATH)},
    {"field": "output directory", "value": results["output_dir"]},
    {"field": "python", "value": sys.version.splitlines()[0]},
    {"field": "platform", "value": platform.platform()},
    {"field": "numpy", "value": np.__version__},
    {"field": "script runtime (s)", "value": results["runtime_seconds"]},
    {"field": "notebook wall time (s)", "value": notebook_elapsed},
    {"field": "seed", "value": config["seed"]},
]
show_table(repro_rows, columns=["field", "value"], title="Reproducibility and runtime log")

config_rows = [
    {"parameter": "dim", "value": config["dim"]},
    {"parameter": "num_layers", "value": config["num_layers"]},
    {"parameter": "num_steps", "value": config["num_steps"]},
    {"parameter": "batch_size", "value": config["batch_size"]},
    {"parameter": "measure_every", "value": config["measure_every"]},
    {"parameter": "momentum", "value": config["momentum"]},
    {"parameter": "Muon LR (fixed)", "value": config["lr_muon"]},
    {"parameter": "SGD LR selection method", "value": lr_search["method"]},
    {"parameter": "chosen SGD LR", "value": lr_search["chosen_lr"]},
    {"parameter": "SGD candidate list", "value": lr_search["candidates"]},
]
show_table(config_rows, columns=["parameter", "value"], title="Configuration used by the script")

artifact_rows = [{"artifact": name, "path": path} for name, path in artifacts.items()]
show_table(artifact_rows, columns=["artifact", "path"], title="Files emitted by this run")


## Problem setup, initial state, and LR calibration

These summaries describe the fixed target matrix, the fixed input batch, the initial near-identity weights, and the coarse SGD learning-rate sweep used by the script.

Two scope notes are important:

1. The SGD LR procedure is **not** a binary search and **not** a symmetric tuning protocol for both optimizers. It simply picks the first stable value from a descending candidate list.
2. Any comparison between the product effective rank and the layers below uses the **mean/min/max across layers**, not a single last-visited layer.


In [ ]:
setup_rows = [
    {"quantity": "target Frobenius norm", "value": problem["target_fro_norm"]},
    {"quantity": "target effective rank", "value": problem["target_effective_rank"]},
    {"quantity": "target condition number", "value": problem["target_condition_number"]},
    {"quantity": "input Frobenius norm", "value": problem["input_fro_norm"]},
    {"quantity": "input mean column norm", "value": problem["input_mean_column_norm"]},
    {"quantity": "initial loss", "value": problem["initial_loss"]},
]
show_table(setup_rows, columns=["quantity", "value"], title="Fixed problem instance")

initial_state = problem["initial_state"]
initial_rows = [
    {"quantity": "initial mean per-layer erank", "value": initial_state["per_layer_erank_mean"]},
    {"quantity": "initial min per-layer erank", "value": initial_state["per_layer_erank_min"]},
    {"quantity": "initial max per-layer erank", "value": initial_state["per_layer_erank_max"]},
    {"quantity": "initial product erank", "value": initial_state["product_effective_rank"]},
    {
        "quantity": "initial product / mean-layer erank",
        "value": initial_state["product_effective_rank"] / initial_state["per_layer_erank_mean"],
    },
    {"quantity": "initial mean per-layer kappa", "value": initial_state["per_layer_kappa_mean"]},
    {"quantity": "initial product kappa", "value": initial_state["product_condition_number"]},
]
show_table(initial_rows, columns=["quantity", "value"], title="Initial spectral state")

show_table(
    problem["initial_state"]["per_layer_rows"],
    columns=["layer", "sigma_min", "sigma_max", "effective_rank", "condition_number"],
    title="Initial per-layer spectral characterization",
)

show_table(
    lr_search["trial_results"],
    columns=["lr", "stable", "failure_step", "initial_loss", "max_loss"],
    title="SGD learning-rate sweep performed by the script",
)


## Spectral trajectories over training

The figure below visualizes the exact arrays returned by `run_experiment.py`.

Interpretation should stay close to what is actually measured:

- panel (a): whether Muon and SGD differ in **per-layer** effective-rank trajectories,
- panel (b): whether the **product** effective rank stays below the layer-level effective rank in this run,
- panel (c): how per-layer condition numbers evolve,
- panel (d): how the product condition number evolves.

This figure does **not** justify any direct claim about asymptotic depth scaling, because the experiment uses only one depth (`L=6`).


In [ ]:
dim = config["dim"]
num_layers = config["num_layers"]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "1.3a-i: Effective rank and conditioning over training\n"
    "Exact arrays returned by run_experiment.py (single-seed, single-depth run)",
    fontsize=14,
    fontweight="bold",
)

ax = axes[0, 0]
for layer in range(num_layers):
    ax.plot(results_sgd["steps"], results_sgd["per_layer_erank"][:, layer], color="tab:blue", alpha=0.25, linewidth=0.8)
for layer in range(num_layers):
    ax.plot(results_muon["steps"], results_muon["per_layer_erank"][:, layer], color="tab:red", alpha=0.25, linewidth=0.8)
ax.plot(results_sgd["steps"], np.mean(results_sgd["per_layer_erank"], axis=1), color="tab:blue", linewidth=2.5, label="SGD mean")
ax.plot(results_muon["steps"], np.mean(results_muon["per_layer_erank"], axis=1), color="tab:red", linewidth=2.5, label="Muon mean")
ax.axhline(dim, color="tab:green", linestyle="--", alpha=0.6, label=f"max erank = {dim}")
ax.set_title("(a) Per-layer effective rank")
ax.set_xlabel("Step")
ax.set_ylabel("Effective rank")
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)

ax = axes[0, 1]
ax.plot(results_sgd["steps"], results_sgd["product_erank"], color="tab:blue", linewidth=2.5, marker="o", markersize=3, label="SGD product")
ax.plot(results_muon["steps"], results_muon["product_erank"], color="tab:red", linewidth=2.5, marker="s", markersize=3, label="Muon product")
ax.axhline(dim, color="tab:green", linestyle="--", alpha=0.6, label=f"max erank = {dim}")
ax.set_title("(b) Product effective rank")
ax.set_xlabel("Step")
ax.set_ylabel("Effective rank")
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)

ax = axes[1, 0]
for layer in range(num_layers):
    ax.semilogy(results_sgd["steps"], results_sgd["per_layer_kappa"][:, layer], color="tab:blue", alpha=0.25, linewidth=0.8)
for layer in range(num_layers):
    ax.semilogy(results_muon["steps"], results_muon["per_layer_kappa"][:, layer], color="tab:red", alpha=0.25, linewidth=0.8)
ax.semilogy(results_sgd["steps"], np.mean(results_sgd["per_layer_kappa"], axis=1), color="tab:blue", linewidth=2.5, label="SGD mean")
ax.semilogy(results_muon["steps"], np.mean(results_muon["per_layer_kappa"], axis=1), color="tab:red", linewidth=2.5, label="Muon mean")
ax.axhline(1.0, color="tab:green", linestyle="--", alpha=0.6, label="orthogonal baseline")
ax.set_title("(c) Per-layer condition number")
ax.set_xlabel("Step")
ax.set_ylabel("Condition number")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)

ax = axes[1, 1]
sgd_product_kappa = results_sgd["product_kappa"].copy()
muon_product_kappa = results_muon["product_kappa"].copy()
sgd_product_kappa[~np.isfinite(sgd_product_kappa)] = np.nan
muon_product_kappa[~np.isfinite(muon_product_kappa)] = np.nan
ax.semilogy(results_sgd["steps"], sgd_product_kappa, color="tab:blue", linewidth=2.5, marker="o", markersize=3, label="SGD product")
ax.semilogy(results_muon["steps"], muon_product_kappa, color="tab:red", linewidth=2.5, marker="s", markersize=3, label="Muon product")
ax.set_title("(d) Product condition number")
ax.set_xlabel("Step")
ax.set_ylabel("Condition number")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle(
    "Per-layer mean/band versus product effective rank\n"
    "Direct view of the notebook's core comparison question",
    fontsize=13,
    fontweight="bold",
)

sgd_mean = np.mean(results_sgd["per_layer_erank"], axis=1)
sgd_min = np.min(results_sgd["per_layer_erank"], axis=1)
sgd_max = np.max(results_sgd["per_layer_erank"], axis=1)
muon_mean = np.mean(results_muon["per_layer_erank"], axis=1)
muon_min = np.min(results_muon["per_layer_erank"], axis=1)
muon_max = np.max(results_muon["per_layer_erank"], axis=1)

ax = axes[0]
ax.set_title("SGD")
ax.fill_between(results_sgd["steps"], sgd_min, sgd_max, color="tab:blue", alpha=0.15, label="layer min-max band")
ax.plot(results_sgd["steps"], sgd_mean, color="tab:blue", linewidth=2.2, label="mean layer erank")
ax.plot(results_sgd["steps"], results_sgd["product_erank"], color="tab:blue", linestyle="--", linewidth=2.2, label="product erank")
ax.axhline(dim, color="tab:green", linestyle=":", alpha=0.6, label=f"max = {dim}")
ax.set_xlabel("Step")
ax.set_ylabel("Effective rank")
ax.set_ylim(bottom=0, top=dim + 2)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)

ax = axes[1]
ax.set_title("Muon")
ax.fill_between(results_muon["steps"], muon_min, muon_max, color="tab:red", alpha=0.15, label="layer min-max band")
ax.plot(results_muon["steps"], muon_mean, color="tab:red", linewidth=2.2, label="mean layer erank")
ax.plot(results_muon["steps"], results_muon["product_erank"], color="tab:red", linestyle="--", linewidth=2.2, label="product erank")
ax.axhline(dim, color="tab:green", linestyle=":", alpha=0.6, label=f"max = {dim}")
ax.set_xlabel("Step")
ax.set_ylabel("Effective rank")
ax.set_ylim(bottom=0, top=dim + 2)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


## Compact report-step tables and final-step summary

The following tables reproduce the key structured summaries returned by the script and make the main comparisons explicit at a small number of report steps.


In [ ]:
report_table = []
for row in report_rows:
    report_table.append(
        {
            "step": row["step"],
            "SGD mean layer erank": row["sgd_per_layer_erank_mean"],
            "Muon mean layer erank": row["muon_per_layer_erank_mean"],
            "SGD product erank": row["sgd_product_erank"],
            "Muon product erank": row["muon_product_erank"],
            "SGD mean layer kappa": row["sgd_per_layer_kappa_mean"],
            "Muon mean layer kappa": row["muon_per_layer_kappa_mean"],
            "SGD product kappa": row["sgd_product_kappa"],
            "Muon product kappa": row["muon_product_kappa"],
            "SGD loss": row["sgd_loss"],
            "Muon loss": row["muon_loss"],
        }
    )
show_table(
    report_table,
    columns=[
        "step",
        "SGD mean layer erank",
        "Muon mean layer erank",
        "SGD product erank",
        "Muon product erank",
        "SGD mean layer kappa",
        "Muon mean layer kappa",
        "SGD product kappa",
        "Muon product kappa",
    ],
    title="Report-step spectral summary",
)

final_rows = [
    {"metric": "final loss", "SGD": final_summary["sgd_final_loss"], "Muon": final_summary["muon_final_loss"]},
    {"metric": "final mean per-layer erank", "SGD": final_summary["sgd_final_per_layer_erank_mean"], "Muon": final_summary["muon_final_per_layer_erank_mean"]},
    {"metric": "final product erank", "SGD": final_summary["sgd_final_product_erank"], "Muon": final_summary["muon_final_product_erank"]},
    {"metric": "final mean per-layer kappa", "SGD": final_summary["sgd_final_per_layer_kappa_mean"], "Muon": final_summary["muon_final_per_layer_kappa_mean"]},
    {"metric": "final product kappa", "SGD": final_summary["sgd_final_product_kappa"], "Muon": final_summary["muon_final_product_kappa"]},
    {"metric": "per-layer erank retention", "SGD": final_summary["sgd_per_layer_erank_retention"], "Muon": final_summary["muon_per_layer_erank_retention"]},
    {"metric": "product erank retention", "SGD": final_summary["sgd_product_erank_retention"], "Muon": final_summary["muon_product_erank_retention"]},
    {"metric": "depth compression = product / mean-layer erank", "SGD": final_summary["sgd_depth_compression"], "Muon": final_summary["muon_depth_compression"]},
    {"metric": "product erank change from step 0", "SGD": final_summary["sgd_product_erank_delta"], "Muon": final_summary["muon_product_erank_delta"]},
]
show_table(final_rows, columns=["metric", "SGD", "Muon"], title="Compact final-step summary")


## Expectations supported vs not supported in this run

The script exposes four explicit checks (T1–T4). The table below reports them exactly, but the interpretation is kept narrowly tied to the current run.


In [ ]:
show_table(
    test_rows_to_table(tests),
    columns=["test", "passed", "description", "observed", "interpretation"],
    title="Expectation checks returned by the script",
)

supported = [label for label in ["T1", "T2", "T3", "T4"] if tests[label]["passed"]]
not_supported = [label for label in ["T1", "T2", "T3", "T4"] if not tests[label]["passed"]]

comparison_note = (
    "lower than" if final_summary["muon_final_per_layer_kappa_mean"] < final_summary["sgd_final_per_layer_kappa_mean"]
    else "higher than or equal to"
)

if tests["T4"]["passed"]:
    t4_sentence = (
        f"T4 is supported here: Muon's final product kappa ({final_summary['muon_final_product_kappa']:.2f}) "
        f"is below SGD's ({final_summary['sgd_final_product_kappa']:.2f})."
    )
else:
    t4_sentence = (
        f"T4 is not supported here: Muon's final product kappa ({final_summary['muon_final_product_kappa']:.2f}) "
        f"exceeds SGD's ({final_summary['sgd_final_product_kappa']:.2f})."
    )

conclusion_lines = [
    "## Calibrated conclusion",
    "",
    f"- **Overall verdict returned by the script:** **{tests['overall']}** ({tests['tests_passed']}/{tests['tests_total']} checks passed).",
    f"- **Supported in this single run:** {', '.join(supported) if supported else 'none'}.",
    f"- **Not supported in this single run:** {', '.join(not_supported) if not_supported else 'none'}.",
    (
        f"- Muon finishes with higher mean per-layer effective rank than SGD "
        f"(**{final_summary['muon_final_per_layer_erank_mean']:.2f}** vs **{final_summary['sgd_final_per_layer_erank_mean']:.2f}**), "
        "which supports the narrow claim that Muon preserved more layer-level spectral spread in this trajectory."
    ),
    (
        f"- The final product effective rank remains below the final mean per-layer effective rank for both optimizers "
        f"(compression factors: SGD **{final_summary['sgd_depth_compression']:.3f}**, "
        f"Muon **{final_summary['muon_depth_compression']:.3f}**). "
        "This supports only the within-run comparison that the product is spectrally narrower than the average layer at the end."
    ),
    (
        f"- However, the product effective rank **increases** from step 0 to step {config['num_steps']} for both optimizers "
        f"(SGD Δ = **{final_summary['sgd_product_erank_delta']:.2f}**, "
        f"Muon Δ = **{final_summary['muon_product_erank_delta']:.2f}**). "
        "So this notebook should **not** describe the current trajectory as showing product-rank collapse over training."
    ),
    (
        f"- Muon's final mean per-layer condition number (**{final_summary['muon_final_per_layer_kappa_mean']:.2f}**) is {comparison_note} "
        f"SGD's (**{final_summary['sgd_final_per_layer_kappa_mean']:.2f}**), but it is still far from the orthogonal ideal κ = 1. "
        "Therefore this run does **not** justify saying that Muon drives per-layer κ toward 1."
    ),
    f"- {t4_sentence}",
    (
        "- Because the experiment uses one seed, one target matrix, one fixed batch, and one depth, "
        "it should be described as an **exploratory mechanistic case study**, not as a direct test of asymptotic depth scaling."
    ),
]
show_markdown("\n".join(conclusion_lines))
